# ITCHI v0.1 frente a v0.2 por posición: mapas de Cristóbal 2020 y Odile 2014

Para cada posición de la trayectoria se dibujan los campos de ITCHI v0.1 y v0.2 lado a lado, con el estilo de mapas de los notebooks anteriores: `configure_map_axis`, mapa centrado en el ciclón, barras de color pequeñas, radio directo (negro) y radio de atribución (magenta) por cuadrante.

| | Lluvia $H_P$ | Viento $H_W$ |
|---|---|---|
| v0.1 | Rampa R3 (Q90 → 0, Q95 → 0.5, Q99 → 1) | Rankine modificado, solo dentro de R34 |
| v0.2 | R4 escalada por R24 y A72 | W06 calibrado a R34, con cola hasta el radio de atribución |

**Entradas.** Los campos que guarda el notebook 07 en `outputs/diagnostics/itchi_v02_snapshots/`. Hay que correr el 07 antes; este notebook no recalcula ITCHI.

**Campos.**
- $P_6$: lluvia de 6 h, con el contorno $P_6 = Q_{95}$ en rojo.
- $H_P$ y $H_W$: peligro por lluvia y por viento.
- $H_{dir}$ y $H_{ind}$: componentes directo e indirecto.
- ITCHI y su clase SIAT.

$H_{ind}$ se obtiene como ITCHI dentro de la región indirecta, porque ahí $H_{dir} = 0$. La fila Δ muestra v0.2 − v0.1; en la clase SIAT es el cambio de clase.

**Salidas.** En `outputs/figures/itchi_v02_posiciones/<caso>/`:
- `rejillas/`: una figura por posición con todos los campos.
- `<campo>/`: un mapa por posición, campo y versión.
- `evento/`: los mapas de `ITCHI_max`.

## 0. Configuración

In [ ]:
import time
import warnings
from dataclasses import replace
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import itchi.itchi_size_helpers as H
from itchi.config import get_project_root, load_default_config
from itchi.plotting import (
    SIAT_DANGER_LABELS,
    MapStyle,
    classify_itchi_to_siat_levels,
    configure_map_axis,
    plot_itchi_field,
    plot_itchi_siat_levels,
)

ROOT = get_project_root()
CFG = load_default_config()
PC = ccrs.PlateCarree()

In [ ]:
CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020", t_map="2020-06-04 00:00"),
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014", t_map="2014-09-15 00:00"),
}
VERSIONS = {"v0.1": "v01", "v0.2": "v02"}      # nombre -> etiqueta en el NetCDF del notebook 07
GRID_FIELDS = ("P6", "H_P", "H_W", "H_dir", "H_ind", "ITCHI", "SIAT")
SINGLE_FIELDS = ("H_P", "H_W", "H_ind", "ITCHI", "SIAT")   # mapas individuales por versión
ADD_DIFF_ROW = True                             # fila v0.2 − v0.1 en las rejillas
SAVE_ALL_GRIDS = True                           # una rejilla por posición
SAVE_ALL_SINGLE = True                          # un mapa por posición, campo y versión
DPI_GRID, DPI_SINGLE = 130, 150

SNAP_HALF_WIDTH_DEG = 14.0
SNAP_STYLE = MapStyle(lon_tick_spacing=8.0, lat_tick_spacing=8.0, grid_spacing=4.0,
                      grid_color="w", grid_alpha=0.6, label_size=10.0)
CBAR = dict(height=0.70, width=0.035, pad=0.03, labelsize=9)   # fracciones del eje del mapa
LABELS = {"P6": "P6 (mm)", "H_P": "H_P", "H_W": "H_W", "H_dir": "H_dir", "H_ind": "H_ind",
          "ITCHI": "ITCHI", "SIAT": "nivel ITCHI"}

CACHE_DIR = ROOT / "outputs" / "diagnostics" / "itchi_v02_snapshots"
CLIM_6H = ROOT / "data" / "processed" / "mswep_v3_6h_percentiles_may_nov.nc"
CLIM_VAR = "p_wet_raw"
FIG_DIR = ROOT / "outputs" / "figures" / "itchi_v02_posiciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
for case in CASES:
    f = CACHE_DIR / f"{case.lower()}_snapshots.nc"
    assert f.exists(), f"falta {f}: corre primero el notebook 07"
print("configuración OK | figuras en", FIG_DIR)

## 1. Campos de v0.1 y v0.2 desde el notebook 07

La trayectoria se lee igual que en los notebooks 05 y 07 (IBTrACS con radios W06 PA21 y ROCLOUD). Se comprueba que sus tiempos coincidan con los del archivo guardado.

In [ ]:
def load_track(case: str) -> pd.DataFrame:
    c = CASES[case]
    ibtracs = ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{c['basin']}.v04r01.nc"
    rocloud = ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{c['basin']}.csv"
    track = H.load_case_track(ibtracs, rocloud, c["name"], c["season"], c["atcf"])
    ok = np.array([H.attribution_radii_km(row, "pa21")[0] is not None for _, row in track.iterrows()])
    return track[ok].reset_index(drop=True)


def read_q95_6h(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    # Q95 de 6 h en la malla del caso, para el contorno P6 = Q95
    with xr.open_dataset(CLIM_6H) as ds:
        q = ds[CLIM_VAR].load()
    q = q.sel(quantile=0.95, method="nearest", drop=True)
    return q.reindex(lat=lat, lon=lon, method="nearest", tolerance=1e-3).values


def load_fields(case: str) -> dict:
    track = load_track(case)
    ds = xr.load_dataset(CACHE_DIR / f"{case.lower()}_snapshots.nc")
    assert np.array_equal(pd.DatetimeIndex(ds["time"].values), pd.DatetimeIndex(track["time"])), \
        f"{case}: los tiempos del archivo no coinciden con la trayectoria; corre el 07 con FORCE_RECOMPUTE = True"
    lat, lon = ds["lat"].values, ds["lon"].values
    m_ind = ds["M_indirect"].values.astype(bool)
    F = {}
    for v, tag in VERSIONS.items():
        itchi = ds[f"ITCHI__{tag}"].values
        F[v] = {"P6": ds["P6"].values, "H_P": ds[f"H_P__{tag}"].values, "H_W": ds[f"H_W__{tag}"].values,
                "H_dir": ds[f"H_dir__{tag}"].values, "H_ind": itchi * m_ind, "ITCHI": itchi}
    lon2d, lat2d = np.meshgrid(lon, lat)
    return dict(track=track, times=pd.DatetimeIndex(ds["time"].values), lon2d=lon2d, lat2d=lat2d,
                q95=read_q95_6h(lat, lon), F=F)


DATA = {case: load_fields(case) for case in CASES}
for case, d in DATA.items():
    print(f"{case}: {len(d['times'])} posiciones, {d['times'][0]:%Y-%m-%d %H} a {d['times'][-1]:%Y-%m-%d %H} UTC")

## 2. Funciones de dibujo

Son las del notebook de atribución con cambios mínimos:
- `snapshot_style`: extensión centrada en el ciclón.
- `overlay_radii`: trayectoria, radio directo y radio de atribución por cuadrante.
- `add_small_colorbar`: barra de color pequeña en un eje insertado.
- `draw_field`: dibuja un campo; en la fila Δ dibuja la diferencia.

In [ ]:
def snapshot_style(row: pd.Series, base: MapStyle = SNAP_STYLE, half_width: float = SNAP_HALF_WIDTH_DEG) -> MapStyle:
    # Extensión centrada en el ciclón, alineada a múltiplos del espaciado de ticks
    step = base.lon_tick_spacing
    lon0 = np.floor((row["lon"] - half_width) / step) * step
    lon1 = np.ceil((row["lon"] + half_width) / step) * step
    lat0 = np.floor((row["lat"] - 0.75 * half_width) / step) * step
    lat1 = np.ceil((row["lat"] + 0.75 * half_width) / step) * step
    return replace(base, lon_extent=(lon0, lon1), lat_extent=(lat0, lat1))


def overlay_radii(ax, track: pd.DataFrame, row: pd.Series) -> None:
    ax.plot(track["lon"], track["lat"], "k-", lw=0.8, transform=PC, zorder=5)
    x, y = H.quadrant_outline(row["lat"], row["lon"], H.direct_radii_km(row))
    ax.plot(x, y, "k-", lw=1.4, transform=ccrs.Geodetic(), zorder=6, label="radio directo (R34)")
    radii, _ = H.attribution_radii_km(row, "pa21")
    if radii is not None:
        x, y = H.quadrant_outline(row["lat"], row["lon"], radii)
        ax.plot(x, y, "m-", lw=1.6, transform=ccrs.Geodetic(), zorder=6, label="atribución (W06 PA21)")
    ax.plot(row["lon"], row["lat"], "k*", ms=10, transform=PC, zorder=7)


def add_small_colorbar(ax, mappable, label: str, ticks=None, ticklabels=None, **opts):
    o = {**CBAR, **opts}
    cax = ax.inset_axes([1.0 + o["pad"], (1.0 - o["height"]) / 2, o["width"], o["height"]])
    cbar = ax.figure.colorbar(mappable, cax=cax, ticks=ticks)
    cbar.set_label(label, fontsize=o["labelsize"])
    cbar.ax.tick_params(labelsize=o["labelsize"] - 1)
    if ticklabels is not None:
        cbar.set_ticklabels(ticklabels)
    return cbar


def siat(x) -> np.ndarray:
    return np.asarray(classify_itchi_to_siat_levels(np.nan_to_num(np.asarray(x, dtype=float))), dtype=float)


def draw_field(ax, d: dict, j: int, field: str, version: str, cbar_opts: dict | None = None):
    # version: "v0.1", "v0.2" o "Δ"
    cbar_opts = cbar_opts or {}
    lon2d, lat2d = d["lon2d"], d["lat2d"]
    if version == "Δ":
        if field == "P6":
            ax.text(0.5, 0.5, "igual en ambas\nversiones", transform=ax.transAxes, ha="center", va="center", fontsize=10)
            return
        if field == "SIAT":
            diff = siat(d["F"]["v0.2"]["ITCHI"][j]) - siat(d["F"]["v0.1"]["ITCHI"][j])
            mesh = ax.pcolormesh(lon2d, lat2d, np.where(diff != 0, diff, np.nan), cmap="RdBu_r", vmin=-3.5, vmax=3.5,
                                 transform=PC, shading="auto", zorder=1)
            add_small_colorbar(ax, mesh, "Δ clase", ticks=range(-3, 4), **cbar_opts)
        else:
            diff = d["F"]["v0.2"][field][j] - d["F"]["v0.1"][field][j]
            mesh = ax.pcolormesh(lon2d, lat2d, np.where(np.abs(diff) > 1e-4, diff, np.nan), cmap="RdBu_r",
                                 vmin=-0.6, vmax=0.6, transform=PC, shading="auto", zorder=1)
            add_small_colorbar(ax, mesh, f"Δ {field}", **cbar_opts)
    elif field == "P6":
        p = d["F"][version]["P6"][j]
        mesh = ax.pcolormesh(lon2d, lat2d, np.where(p >= 0.1, p, np.nan), cmap="YlGnBu", shading="auto",
                             transform=PC, zorder=1)
        add_small_colorbar(ax, mesh, LABELS["P6"], **cbar_opts)
        ax.contour(lon2d, lat2d, np.nan_to_num(p - d["q95"], nan=-1.0), levels=[0.0], colors="r",
                   linewidths=0.7, transform=PC, zorder=5)
    elif field == "SIAT":
        mesh = plot_itchi_siat_levels(d["F"][version]["ITCHI"][j], ax, lon=lon2d, lat=lat2d, crs=PC,
                                      add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, LABELS["SIAT"], ticks=np.arange(0.5, 6.5, 1.0),
                           ticklabels=[lab.split(" ", 1)[0] for lab in SIAT_DANGER_LABELS], **cbar_opts)
    else:
        v = np.asarray(d["F"][version][field][j], dtype=float)
        mesh = plot_itchi_field(np.where(v > 0, v, np.nan), ax, lon=lon2d, lat=lat2d, crs=PC,
                                add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, LABELS[field], **cbar_opts)

In [ ]:
def plot_position_grid(case: str, t, fields=GRID_FIELDS, diff_row: bool = ADD_DIFF_ROW):
    # Rejilla versiones (filas) x campos (columnas) en una posición
    d = DATA[case]
    j = d["times"].get_loc(pd.Timestamp(t))
    row = d["track"].iloc[j]
    style = snapshot_style(row)
    rows = list(VERSIONS) + (["Δ"] if diff_row else [])
    fig, axes = plt.subplots(len(rows), len(fields), figsize=(4.6 * len(fields), 3.9 * len(rows)),
                             subplot_kw={"projection": PC}, squeeze=False)
    fig.subplots_adjust(wspace=0.45, hspace=0.25)
    for i, version in enumerate(rows):
        for k, field in enumerate(fields):
            ax = configure_map_axis(axes[i, k], crs=PC, style=style)
            draw_field(ax, d, j, field, version)
            overlay_radii(ax, d["track"], row)
            ax.set_title(f"{field} · {version if version != 'Δ' else 'v0.2 − v0.1'}", fontsize=11)
    axes[0, 0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{case} {d['times'][j]:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt", fontsize=14, y=0.995)
    return fig


def plot_position_field(case: str, t, version: str, field: str, half_width: float = SNAP_HALF_WIDTH_DEG,
                        figsize=(9, 7), cbar_opts=None):
    # Un mapa a detalle: una posición, una versión (o "Δ"), un campo
    d = DATA[case]
    j = d["times"].get_loc(pd.Timestamp(t))
    row = d["track"].iloc[j]
    base = replace(SNAP_STYLE, label_size=12.0, tick_size=7.0)
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": PC})
    configure_map_axis(ax, crs=PC, style=snapshot_style(row, base, half_width))
    draw_field(ax, d, j, field, version, cbar_opts={"labelsize": 11, **(cbar_opts or {})})
    overlay_radii(ax, d["track"], row)
    ax.legend(fontsize=9, loc="lower left")
    name = "v0.2 − v0.1" if version == "Δ" else f"ITCHI {version}"
    ax.set_title(f"{field} · {name}\n{case} {d['times'][j]:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt", fontsize=12)
    return fig

## 3. Ejemplo en `t_map`

Rejilla completa y un par de mapas a detalle en la posición de referencia de cada caso. Para ver otra posición, cambia `t`.

In [ ]:
for case, c in CASES.items():
    t = pd.Timestamp(c["t_map"])
    fig = plot_position_grid(case, t)
    fig.savefig(FIG_DIR / f"{case.lower()}_rejilla_{t:%Y%m%d%H}.png", dpi=DPI_GRID, bbox_inches="tight")
    plt.show()

In [ ]:
case = "CRISTOBAL2020"
t = pd.Timestamp(CASES[case]["t_map"])
fig = plot_position_field(case, t, "v0.1", "SIAT")
fig = plot_position_field(case, t, "v0.2", "SIAT")
fig = plot_position_field(case, t, "Δ", "ITCHI")

## 4. Todas las posiciones

Guarda una rejilla por posición en `rejillas/` y, si `SAVE_ALL_SINGLE` es True, un mapa por posición, campo de `SINGLE_FIELDS` y versión (v0.1, v0.2 y Δ). Son muchas figuras: con ~40 posiciones por caso, cuenta unos minutos por caso. Las figuras se cierran al guardarse para no llenar la memoria.

Los nombres empiezan con la fecha (`AAAAMMDDHH`), así que al ordenarlos quedan en secuencia y sirven como cuadros de una animación.

In [ ]:
# for case, d in DATA.items():
#     out = FIG_DIR / case.lower()
#     t0, n = time.time(), 0
#     for t in d["times"]:
#         if SAVE_ALL_GRIDS:
#             (out / "rejillas").mkdir(parents=True, exist_ok=True)
#             fig = plot_position_grid(case, t)
#             fig.savefig(out / "rejillas" / f"{t:%Y%m%d%H}.png", dpi=DPI_GRID, bbox_inches="tight")
#             plt.close(fig); n += 1
#         if SAVE_ALL_SINGLE:
#             for field in SINGLE_FIELDS:
#                 (out / field).mkdir(parents=True, exist_ok=True)
#                 for version in (*VERSIONS, "Δ"):
#                     fig = plot_position_field(case, t, version, field)
#                     tag = "dif" if version == "Δ" else VERSIONS[version]
#                     fig.savefig(out / field / f"{t:%Y%m%d%H}_{tag}.png", dpi=DPI_SINGLE, bbox_inches="tight")
#                     plt.close(fig); n += 1
#         print(f"\r{case}: {t:%Y-%m-%d %H} UTC | {n} figuras | {time.time() - t0:.0f} s", end="", flush=True)
#     print()

## 5. Mapas del evento: `ITCHI_max`

Máximo temporal de ITCHI en cada píxel para v0.1, v0.2 y su diferencia, en el mismo estilo, sobre todo el dominio del caso. Son los mapas de resumen para la presentación.

In [ ]:
def event_style(d: dict, emax: dict, pad: float = 2.0, step: float = 8.0) -> MapStyle:
    # Extensión de la huella del evento (ITCHI_max > 0 en alguna versión), no de toda la trayectoria
    m = np.zeros(d["lon2d"].shape, bool)
    for v in emax.values():
        m |= np.nan_to_num(v) > 0
    lon, lat = d["lon2d"][m], d["lat2d"][m]
    lon0 = np.floor((lon.min() - pad) / step) * step; lon1 = np.ceil((lon.max() + pad) / step) * step
    lat0 = np.floor((lat.min() - pad) / step) * step; lat1 = np.ceil((lat.max() + pad) / step) * step
    return replace(SNAP_STYLE, lon_extent=(lon0, lon1), lat_extent=(lat0, lat1))


for case, d in DATA.items():
    out = FIG_DIR / case.lower() / "evento"
    out.mkdir(parents=True, exist_ok=True)
    emax = {v: np.nanmax(d["F"][v]["ITCHI"], axis=0) for v in VERSIONS}
    style = event_style(d, emax)
    fig, axes = plt.subplots(2, 3, figsize=(17, 9.5), subplot_kw={"projection": PC})
    fig.subplots_adjust(wspace=0.4, hspace=0.2)
    for k, v in enumerate(VERSIONS):
        ax = configure_map_axis(axes[0, k], crs=PC, style=style)
        mesh = plot_itchi_field(np.where(emax[v] > 0, emax[v], np.nan), ax, lon=d["lon2d"], lat=d["lat2d"], crs=PC,
                                add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, "ITCHI_max")
        ax.set_title(f"ITCHI_max · {v}")
        ax = configure_map_axis(axes[1, k], crs=PC, style=style)
        mesh = plot_itchi_siat_levels(emax[v], ax, lon=d["lon2d"], lat=d["lat2d"], crs=PC, add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, "nivel ITCHI", ticks=np.arange(0.5, 6.5, 1.0),
                           ticklabels=[lab.split(" ", 1)[0] for lab in SIAT_DANGER_LABELS])
        ax.set_title(f"Clase SIAT de ITCHI_max · {v}")
    diff = emax["v0.2"] - emax["v0.1"]
    dcls = siat(emax["v0.2"]) - siat(emax["v0.1"])
    for i, (fld, lab, vm) in enumerate(((diff, "Δ ITCHI_max", 0.6), (dcls, "Δ clase", 3.5))):
        ax = configure_map_axis(axes[i, 2], crs=PC, style=style)
        mesh = ax.pcolormesh(d["lon2d"], d["lat2d"], np.where(np.abs(fld) > 1e-4, fld, np.nan), cmap="RdBu_r",
                             vmin=-vm, vmax=vm, transform=PC, shading="auto", zorder=1)
        add_small_colorbar(ax, mesh, lab, ticks=range(-3, 4) if i else None)
        ax.set_title(f"{lab} (v0.2 − v0.1)")
    for ax in axes.ravel():
        ax.plot(d["track"]["lon"], d["track"]["lat"], "k.-", lw=0.8, ms=2.5, transform=PC, zorder=5)
    fig.suptitle(f"{case}: resumen del evento", fontsize=14, y=0.98)
    fig.savefig(out / "itchi_max_v01_v02.png", dpi=200, bbox_inches="tight")
print("figuras en", FIG_DIR)